# EVALUATING AGRICLIP + SAM2 PIPELINE ON MODERN FRUIT ORCHARDS Dataset

In [4]:
import os
import sys
import json
import math
import time
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image

import torch
import torch.nn.functional as F
from torchvision import transforms

from tqdm.auto import tqdm

In [5]:
PROJECT_ROOT = Path(
    "/home/zahranadeem/thesis"
)

MFO_ROOT = (
    PROJECT_ROOT / "MFO"
)

MODEL_NAME = (
    "facebook/sam2.1-hiera-tiny"
)

# AgriCLIP
AGRICLIP_ROOT = (
    PROJECT_ROOT
    / "agriclip_sam2"
    / "AgriCLIP"
)

ALIGNMENT_DIR = (
    AGRICLIP_ROOT
    / "AgriCLIP_alignment"
)

ALIGNER_PATH = (
    PROJECT_ROOT
    / "vegann_aligner.pth"
)

# Fine-tuned SAM2 decoder
SAM2_CHECKPOINT = (
    PROJECT_ROOT
    / "sam2_vegann_checkpoints"
    / "best_sam2_vegann_mask_decoder.pt"
)

# Output
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "agriclip_attention_guided_sam2_mfo"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project:", PROJECT_ROOT)
print("MFO:", MFO_ROOT)
print("AgriCLIP:", AGRICLIP_ROOT)
print("Aligner:", ALIGNER_PATH)
print("SAM2 checkpoint:", SAM2_CHECKPOINT)
print("Output:", OUTPUT_DIR)

Project: /home/zahranadeem/thesis
MFO: /home/zahranadeem/thesis/MFO
AgriCLIP: /home/zahranadeem/thesis/agriclip_sam2/AgriCLIP
Aligner: /home/zahranadeem/thesis/vegann_aligner.pth
SAM2 checkpoint: /home/zahranadeem/thesis/sam2_vegann_checkpoints/best_sam2_vegann_mask_decoder.pt
Output: /home/zahranadeem/thesis/results/agriclip_attention_guided_sam2_mfo


In [7]:
print("MFO exists:", MFO_ROOT.exists())

if MFO_ROOT.exists():

    print("\nMFO contents:\n")

    for item in sorted(MFO_ROOT.iterdir()):
        print(item)

else:

    print(
        "MFO directory not found."
    )

MFO exists: True

MFO contents:

/home/zahranadeem/thesis/MFO/.git
/home/zahranadeem/thesis/MFO/LICENSE
/home/zahranadeem/thesis/MFO/Paper.pdf
/home/zahranadeem/thesis/MFO/README.md
/home/zahranadeem/thesis/MFO/image


In [8]:
print(
    "\nFiles/directories inside MFO:\n"
)

for path in MFO_ROOT.rglob("*"):

    if path.is_file():

        print(
            path.relative_to(MFO_ROOT)
        )


Files/directories inside MFO:

README.md
Paper.pdf
LICENSE
image/synth_data1.png
image/real_data.png
.git/description
.git/packed-refs
.git/HEAD
.git/config
.git/index
.git/hooks/pre-rebase.sample
.git/hooks/pre-commit.sample
.git/hooks/update.sample
.git/hooks/pre-merge-commit.sample
.git/hooks/pre-receive.sample
.git/hooks/pre-push.sample
.git/hooks/post-update.sample
.git/hooks/push-to-checkout.sample
.git/hooks/commit-msg.sample
.git/hooks/prepare-commit-msg.sample
.git/hooks/pre-applypatch.sample
.git/hooks/fsmonitor-watchman.sample
.git/hooks/applypatch-msg.sample
.git/info/exclude
.git/logs/HEAD
.git/logs/refs/remotes/origin/HEAD
.git/logs/refs/heads/main
.git/refs/remotes/origin/HEAD
.git/refs/heads/main
.git/objects/pack/pack-62d54ac55946ad1ad32031862b112be322c8b9e0.pack
.git/objects/pack/pack-62d54ac55946ad1ad32031862b112be322c8b9e0.idx


In [9]:
#Device 
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )



Device: cuda
GPU: NVIDIA GeForce RTX 4090


In [10]:
#DINO preprocessing 
IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]

dino_transform = transforms.Compose([

    transforms.Resize(
        (224, 224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD
    ),
])

In [11]:
#Load DINO 
dino_model = torch.hub.load(
    "facebookresearch/dino:main",
    "dino_resnet50",
    pretrained=True,
)

dino_model.eval()
dino_model.to(device)

for parameter in dino_model.parameters():
    parameter.requires_grad = False

print("DINO loaded.")
print(
    "Parameters:",
    sum(
        p.numel()
        for p in dino_model.parameters()
    )
)

Using cache found in /home/zahranadeem/.cache/torch/hub/facebookresearch_dino_main
/home/zahranadeem/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zahranadeem/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


DINO loaded.
Parameters: 23508032


In [12]:
#Import Agriclip modules
print(
    "TextToConcept:",
    (
        ALIGNMENT_DIR /
        "TextToConcept.py"
    ).exists()
)

print(
    "LinearAligner:",
    (
        ALIGNMENT_DIR /
        "LinearAligner.py"
    ).exists()
)

TextToConcept: True
LinearAligner: True


In [14]:
if str(ALIGNMENT_DIR) not in sys.path:

    sys.path.insert(
        0,
        str(ALIGNMENT_DIR)
    )

from TextToConcept import TextToConcept
from LinearAligner import LinearAligner

print(
    "AgriCLIP modules imported successfully."
)

AgriCLIP modules imported successfully.


In [15]:
#Initializing AgriCLIP textToConcepr 
text_to_concept = TextToConcept(
    dino_model,
    "dino_resnet50",
)

text_to_concept.load_linear_aligner(
    str(ALIGNER_PATH)
)

aligner = (
    text_to_concept.linear_aligner
)

print(
    "VegAnn-trained aligner loaded."
)

print(
    "W shape:",
    tuple(aligner.W.shape)
)

print(
    "b shape:",
    tuple(aligner.b.shape)
)

VegAnn-trained aligner loaded.
W shape: (512, 2048)
b shape: (512,)


In [16]:
#DINO layer 4 hook 
captured_feature_map = None


def capture_layer4_output(
    module,
    inputs,
    output
):

    global captured_feature_map

    captured_feature_map = (
        output.detach()
    )


hook_handle = (
    dino_model.layer4.register_forward_hook(
        capture_layer4_output
    )
)

print(
    "DINO layer4 hook registered."
)

DINO layer4 hook registered.


In [17]:
#Extracting DINO features 
@torch.no_grad()
def extract_dino_patch_features(
    pil_image
):

    global captured_feature_map

    captured_feature_map = None

    image_tensor = (
        dino_transform(
            pil_image
        )
        .unsqueeze(0)
        .to(device)
    )

    _ = dino_model(
        image_tensor
    )

    if captured_feature_map is None:

        raise RuntimeError(
            "Layer-4 hook did not capture "
            "a feature map."
        )

    if captured_feature_map.ndim != 4:

        raise RuntimeError(
            "Expected feature map "
            "(B,C,H,W), got "
            f"{captured_feature_map.shape}"
        )

    _, channels, grid_h, grid_w = (
        captured_feature_map.shape
    )

    patch_features = (
        captured_feature_map[0]
        .permute(1, 2, 0)
        .reshape(
            grid_h * grid_w,
            channels
        )
        .contiguous()
    )

    return (
        patch_features,
        grid_h,
        grid_w
    )

In [18]:
#Projecting in AgriCLIP concept space 
@torch.no_grad()
def align_patch_features(
    patch_features
):

    patch_features = (
        patch_features
        .float()
        .to(device)
    )

    aligned_features = (
        aligner.get_aligned_representation(
            patch_features
        )
    )

    aligned_features = F.normalize(
        aligned_features,
        dim=1
    )

    return aligned_features

In [19]:
#Vegetation Prompts 
VEGETATION_PROMPTS = [

    "A crop field populated by healthy plants, captured from a nadir view using a handheld camera.",

    "An agricultural scene showing healthy crops growing in the field, captured from a nadir view using a handheld camera.",

    "A field image containing healthy crop vegetation, captured from a nadir view using a handheld camera.",

    "An aerial agricultural image of dense green crop plants, captured from a nadir view using a handheld camera.",

    "Healthy crop canopy covering the field, captured from a nadir view using a handheld camera.",

    "Agricultural vegetation viewed from above, captured from a nadir view using a handheld camera.",
]

for prompt in VEGETATION_PROMPTS:

    print(
        "+",
        prompt
    )

+ A crop field populated by healthy plants, captured from a nadir view using a handheld camera.
+ An agricultural scene showing healthy crops growing in the field, captured from a nadir view using a handheld camera.
+ A field image containing healthy crop vegetation, captured from a nadir view using a handheld camera.
+ An aerial agricultural image of dense green crop plants, captured from a nadir view using a handheld camera.
+ Healthy crop canopy covering the field, captured from a nadir view using a handheld camera.
+ Agricultural vegetation viewed from above, captured from a nadir view using a handheld camera.


In [20]:
#Encode vegetation prompts 
@torch.no_grad()
def encode_prompt_set(
    prompts
):

    embeddings = []

    for prompt in prompts:

        embedding = (
            text_to_concept.encode_text(
                [[prompt]]
            )
        )

        if isinstance(
            embedding,
            np.ndarray
        ):

            embedding = (
                torch.from_numpy(
                    embedding
                )
            )

        if not torch.is_tensor(
            embedding
        ):

            raise TypeError(
                "Unsupported embedding type: "
                f"{type(embedding)}"
            )

        embedding = (
            embedding
            .detach()
            .float()
            .to(device)
            .reshape(-1)
        )

        embeddings.append(
            embedding
        )

    embeddings = torch.stack(
        embeddings,
        dim=0
    )

    embeddings = F.normalize(
        embeddings,
        dim=1
    )

    mean_embedding = embeddings.mean(
        dim=0
    )

    mean_embedding = F.normalize(
        mean_embedding,
        dim=0
    )

    return (
        embeddings,
        mean_embedding
    )


(
    vegetation_text_embeddings,
    vegetation_embedding
) = encode_prompt_set(
    VEGETATION_PROMPTS
)

print(
    "Text embeddings:",
    vegetation_text_embeddings.shape
)

print(
    "Mean embedding:",
    vegetation_embedding.shape
)

Text embeddings: torch.Size([6, 512])
Mean embedding: torch.Size([512])


In [21]:
#generate AgriCLIP box 
@torch.no_grad()
def generate_agriclip_box(
    image,
    top_k=8,
    margin=25
):

    patch_features, grid_h, grid_w = (
        extract_dino_patch_features(
            image
        )
    )

    aligned_features = (
        align_patch_features(
            patch_features
        )
    )

    # Patch-to-text similarity
    scores = (
        aligned_features
        @ vegetation_embedding
    )

    scores_np = (
        scores.detach()
        .cpu()
        .numpy()
    )

    top_k = min(
        top_k,
        len(scores_np)
    )

    topk_indices = (
        np.argsort(
            scores_np
        )[::-1][:top_k]
    )

    image_width, image_height = (
        image.size
    )

    patch_width = (
        image_width / grid_w
    )

    patch_height = (
        image_height / grid_h
    )

    boxes = []

    for idx in topk_indices:

        row = (
            idx // grid_w
        )

        col = (
            idx % grid_w
        )

        x1 = int(
            col * patch_width
        )

        y1 = int(
            row * patch_height
        )

        x2 = int(
            (col + 1)
            * patch_width
        )

        y2 = int(
            (row + 1)
            * patch_height
        )

        boxes.append(
            [
                x1,
                y1,
                x2,
                y2
            ]
        )

    boxes = np.array(
        boxes,
        dtype=np.int32
    )

    # Merge top-K patches
    xmin = boxes[:, 0].min()
    ymin = boxes[:, 1].min()

    xmax = boxes[:, 2].max()
    ymax = boxes[:, 3].max()

    # Add margin
    xmin = max(
        0,
        xmin - margin
    )

    ymin = max(
        0,
        ymin - margin
    )

    xmax = min(
        image_width - 1,
        xmax + margin
    )

    ymax = min(
        image_height - 1,
        ymax + margin
    )

    agriclip_box = np.array(
        [
            xmin,
            ymin,
            xmax,
            ymax
        ],
        dtype=np.int32
    )

    return (
        agriclip_box,
        boxes,
        scores_np,
        topk_indices
    )

In [ ]:
#Testing Agriclip on one MFO image 
TEST_IMAGE = Path(
    "PUT_ACTUAL_MFO_IMAGE_PATH_HERE"
)

image = Image.open(
    TEST_IMAGE
).convert("RGB")

print(
    "Image size:",
    image.size
)

In [ ]:
#Test AgriCLIP box 
(
    agriclip_box,
    boxes,
    scores,
    topk_indices
) = generate_agriclip_box(
    image,
    top_k=8,
    margin=25
)

print(
    "AgriCLIP box:",
    agriclip_box
)

print(
    "Top-K indices:",
    topk_indices
)

print(
    "Top-K scores:",
    scores[topk_indices]
)


In [ ]:
#Visualize
fig, ax = plt.subplots(
    figsize=(8, 8)
)

ax.imshow(image)

# Top-K patches
for box in boxes:

    x1, y1, x2, y2 = box

    ax.add_patch(
        patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            linewidth=1.5
        )
    )

# Final merged box
x1, y1, x2, y2 = (
    agriclip_box
)

ax.add_patch(
    patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=3
    )
)

ax.set_title(
    "MFO: AgriCLIP Top-K Patches + Generated Box"
)

ax.axis("off")

plt.tight_layout()
plt.show() 


In [ ]:
#Load SAM2 
from transformers import (
    Sam2Model,
    Sam2Processor,
)

sam2_processor = (
    Sam2Processor.from_pretrained(
        MODEL_NAME
    )
)

sam2_model = (
    Sam2Model.from_pretrained(
        MODEL_NAME
    )
)

print(
    "SAM2 loaded."
)



In [ ]:
sam2_checkpoint = torch.load(
    SAM2_CHECKPOINT,
    map_location="cpu"
)

print(
    "Checkpoint type:",
    type(sam2_checkpoint)
)

if (
    isinstance(
        sam2_checkpoint,
        dict
    )
    and
    "mask_decoder_state_dict"
    in sam2_checkpoint
):

    mask_decoder_state_dict = (
        sam2_checkpoint[
            "mask_decoder_state_dict"
        ]
    )

elif isinstance(
    sam2_checkpoint,
    dict
):

    mask_decoder_state_dict = (
        sam2_checkpoint
    )

else:

    raise TypeError(
        "Unsupported checkpoint structure."
    )

load_result = (
    sam2_model.mask_decoder
    .load_state_dict(
        mask_decoder_state_dict,
        strict=True
    )
)

sam2_model = (
    sam2_model
    .to(device)
)

sam2_model.eval()

print(
    "Fine-tuned SAM2 decoder loaded."
)

print(load_result)

In [ ]:
#Predict 
@torch.no_grad()
def sam2_predict_box(
    image,
    box,
    threshold=0.5
):

    sam2_inputs = (
        sam2_processor(
            images=image,
            input_boxes=[
                [box.tolist()]
            ],
            return_tensors="pt"
        )
    )

    sam2_inputs = {
        k: (
            v.to(device)
            if torch.is_tensor(v)
            else v
        )
        for k, v in sam2_inputs.items()
    }

    outputs = sam2_model(
        **sam2_inputs
    )

    best_candidate_index = int(
        outputs.iou_scores[
            0,
            0
        ].argmax().item()
    )

    logits = outputs.pred_masks[
        0,
        0,
        best_candidate_index
    ]

    probability = torch.sigmoid(
        logits
    )

    original_height = int(
        sam2_inputs[
            "original_sizes"
        ][0, 0].item()
    )

    original_width = int(
        sam2_inputs[
            "original_sizes"
        ][0, 1].item()
    )

    probability = F.interpolate(
        probability[None, None],
        size=(
            original_height,
            original_width
        ),
        mode="bilinear",
        align_corners=False
    )[0, 0]

    probability_np = (
        probability
        .detach()
        .cpu()
        .numpy()
    )

    pred_mask = (
        probability_np >= threshold
    ).astype(np.uint8)

    confidence = float(
        outputs.iou_scores[
            0,
            0,
            best_candidate_index
        ].item()
    )

    return (
        pred_mask,
        probability_np,
        confidence
    )

In [ ]:
#Merics 
def compute_binary_metrics(
    prediction,
    ground_truth
):

    prediction = (
        prediction.astype(bool)
    )

    ground_truth = (
        ground_truth.astype(bool)
    )

    intersection = np.logical_and(
        prediction,
        ground_truth
    ).sum()

    union = np.logical_or(
        prediction,
        ground_truth
    ).sum()

    pred_pixels = (
        prediction.sum()
    )

    gt_pixels = (
        ground_truth.sum()
    )

    iou = (
        intersection / union
        if union > 0
        else 1.0
    )

    dice_denominator = (
        pred_pixels + gt_pixels
    )

    dice = (
        2.0 * intersection
        / dice_denominator
        if dice_denominator > 0
        else 1.0
    )

    precision = (
        intersection / pred_pixels
        if pred_pixels > 0
        else 0.0
    )

    recall = (
        intersection / gt_pixels
        if gt_pixels > 0
        else 0.0
    )

    return {
        "iou": float(iou),
        "dice": float(dice),
        "precision": float(precision),
        "recall": float(recall),
        "intersection": int(
            intersection
        ),
        "union": int(union),
        "pred_pixels": int(
            pred_pixels
        ),
        "gt_pixels": int(
            gt_pixels
        )
    }

In [ ]:
#Testing pipeline on 1 image 
image = Image.open(
    TEST_IMAGE
).convert("RGB")

# Replace with your actual mask loading
gt_mask = ...

(
    agriclip_box,
    boxes,
    scores,
    topk_indices
) = generate_agriclip_box(
    image,
    top_k=8,
    margin=25
)

(
    pred_mask,
    probability,
    confidence
) = sam2_predict_box(
    image,
    agriclip_box,
    threshold=0.5
)

metrics = compute_binary_metrics(
    pred_mask,
    gt_mask
)

print(
    "AgriCLIP box:",
    agriclip_box
)

print(
    "SAM2 confidence:",
    confidence
)

for name, value in metrics.items():

    if isinstance(
        value,
        float
    ):

        print(
            f"{name:>15}: "
            f"{value:.4f}"
        )

    else:

        print(
            f"{name:>15}: "
            f"{value}"
        )

In [ ]:
#Visualize complete result 
fig, ax = plt.subplots(
    2,
    3,
    figsize=(16, 10)
)

# Input
ax[0, 0].imshow(image)
ax[0, 0].set_title(
    "MFO Input"
)
ax[0, 0].axis("off")

# AgriCLIP box
ax[0, 1].imshow(image)

x1, y1, x2, y2 = (
    agriclip_box
)

ax[0, 1].add_patch(
    patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        fill=False,
        linewidth=3
    )
)

ax[0, 1].set_title(
    "AgriCLIP-generated box"
)

ax[0, 1].axis("off")

# Probability
ax[0, 2].imshow(
    probability,
    cmap="jet"
)

ax[0, 2].set_title(
    f"SAM2 probability\n"
    f"confidence={confidence:.3f}"
)

ax[0, 2].axis("off")

# GT
ax[1, 0].imshow(
    gt_mask,
    cmap="gray"
)

ax[1, 0].set_title(
    "MFO Ground Truth"
)

ax[1, 0].axis("off")

# Prediction
ax[1, 1].imshow(
    pred_mask,
    cmap="gray"
)

ax[1, 1].set_title(
    f"SAM2 Prediction\n"
    f"IoU={metrics['iou']:.4f}"
)

ax[1, 1].axis("off")

# Overlay
ax[1, 2].imshow(image)

ax[1, 2].imshow(
    pred_mask,
    cmap="jet",
    alpha=0.4
)

ax[1, 2].set_title(
    f"Prediction Overlay\n"
    f"Dice={metrics['dice']:.4f}"
)

ax[1, 2].axis("off")

plt.tight_layout()
plt.show()